In [1]:
import numpy as npf
from pathlib import Path

In [2]:
# Load model and validation data from ../model
import numpy as np
import tensorflow as tf
from pathlib import Path

MODEL_DIR = Path('..') / 'model'
print('Looking for model and data in', MODEL_DIR)
keras_files = sorted(MODEL_DIR.glob('*.keras')) + sorted(MODEL_DIR.glob('*.h5'))
tflite_files = sorted(MODEL_DIR.glob('*.tflite'))
model = None
interpreter = None
if keras_files:
    model_path = keras_files[0]
    print('Loading Keras model:', model_path)
    model = tf.keras.models.load_model(str(model_path))
elif tflite_files:
    tflite_path = tflite_files[0]
    print('Loading TFLite model (interpreter):', tflite_path)
    interpreter = tf.lite.Interpreter(model_path=str(tflite_path))
    interpreter.allocate_tensors()
else:
    print('No .keras/.h5/.tflite model found in', MODEL_DIR)

# Load validation arrays if present
X_val_path = MODEL_DIR / 'X_val.npy'
y_val_path = MODEL_DIR / 'y_val.npy'
X_val = np.load(X_val_path) if X_val_path.exists() else None
y_val = np.load(y_val_path) if y_val_path.exists() else None
print('X_val shape:', None if X_val is None else X_val.shape)
print('y_val shape:', None if y_val is None else y_val.shape)

# Prepare and save a single quantized sample for the testbench (int16 Q15 style)
if X_val is not None:
    X_sample = X_val[0:1]
    SAMPLE_Q = np.clip(np.round(X_sample.reshape(-1) * (2**15)), -32768, 32767).astype(np.int16)
    np.savetxt('img_q_int16_from_X_val.txt', SAMPLE_Q, fmt='%d')
    print('Saved sample image as img_q_int16_from_X_val.txt')
else:
    X_sample = None


2025-12-09 11:51:41.837852: I tensorflow/core/util/port.cc:113] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2025-12-09 11:51:42.287130: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:9261] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2025-12-09 11:51:42.287357: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:607] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2025-12-09 11:51:42.350180: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1515] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2025-12-09 11:51:42.481332: I tensorflow/core/platform/cpu_feature_guar

Looking for model and data in ../model
Loading Keras model: ../model/Hanni_Benarib_model.keras


2025-12-09 11:51:46.813017: I external/local_xla/xla/stream_executor/cuda/cuda_executor.cc:901] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero. See more at https://github.com/torvalds/linux/blob/v6.0/Documentation/ABI/testing/sysfs-bus-pci#L344-L355
2025-12-09 11:51:47.273513: I external/local_xla/xla/stream_executor/cuda/cuda_executor.cc:901] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero. See more at https://github.com/torvalds/linux/blob/v6.0/Documentation/ABI/testing/sysfs-bus-pci#L344-L355
2025-12-09 11:51:47.276761: I external/local_xla/xla/stream_executor/cuda/cuda_executor.cc:901] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero. See more at https://github.com/torvalds/linux/blob/v6.0/Documentation/ABI/testing/sysfs-bus-pci#L344-

X_val shape: (1651, 98, 12, 1)
y_val shape: (1651, 12)
Saved sample image as img_q_int16_from_X_val.txt


In [3]:
ker = np.loadtxt("conv1_kernel_000.txt", dtype=np.int16)  # shape (9,)

print("constant K0 : kernel3x3_t := (")
for i, v in enumerate(ker):
    sep = "," if i < 8 else ""
    print(f"  {i} => to_signed({int(v)}, 16){sep}")
print(");")

constant K0 : kernel3x3_t := (
  0 => to_signed(-33, 16),
  1 => to_signed(-28, 16),
  2 => to_signed(13, 16),
  3 => to_signed(-32, 16),
  4 => to_signed(-28, 16),
  5 => to_signed(-5, 16),
  6 => to_signed(-31, 16),
  7 => to_signed(-56, 16),
  8 => to_signed(-57, 16)
);


In [4]:
import numpy as np

# === Paramètres à adapter si besoin ===
N_FILTERS   = 32                 # taille de la banque (0..31)
WIDTH       = 16                 # largeur des poids en VHDL (signed(WIDTH-1 downto 0))
CONST_NAME  = "K_BANK32_L0"      # nom de la constante VHDL
TYPE_NAME   = "kernel3x3_bank32_t"
k = 0                            # indice du groupe de kernels à charger (0..31)        

# Dossier et pattern des fichiers de kernels
FILE_PATTERN = "conv1_kernel_{:03d}.txt"  # conv1_kernel_000.txt, 001, ...

def load_kernel(idx: int) -> np.ndarray:
    """Charge un kernel 3x3 depuis un fichier texte (9 int16)."""
    fname = FILE_PATTERN.format(idx)
    ker = np.loadtxt(fname, dtype=np.int16)
    if ker.shape[0] != 9:
        raise ValueError(f"Kernel {fname} ne contient pas 9 valeurs (shape = {ker.shape})")
    return ker

print(f"constant {CONST_NAME} : {TYPE_NAME} := (")
for f in range(k*N_FILTERS,(k+1)*N_FILTERS):
    ker = load_kernel(f)
    print(f"  {f%32} => (")
    for i, v in enumerate(ker):
        sep = "," if i < 8 else ""
        print(f"    {i} => to_signed({int(v)}, {WIDTH}){sep}")
    # virgule après le filtre sauf pour le dernier
    end_sep = "," if f < N_FILTERS - 1 else ""
    print(f"  ){end_sep}")
print(");")


constant K_BANK32_L0 : kernel3x3_bank32_t := (
  0 => (
    0 => to_signed(-33, 16),
    1 => to_signed(-28, 16),
    2 => to_signed(13, 16),
    3 => to_signed(-32, 16),
    4 => to_signed(-28, 16),
    5 => to_signed(-5, 16),
    6 => to_signed(-31, 16),
    7 => to_signed(-56, 16),
    8 => to_signed(-57, 16)
  ),
  1 => (
    0 => to_signed(19, 16),
    1 => to_signed(7, 16),
    2 => to_signed(-65, 16),
    3 => to_signed(13, 16),
    4 => to_signed(-4, 16),
    5 => to_signed(30, 16),
    6 => to_signed(-33, 16),
    7 => to_signed(-56, 16),
    8 => to_signed(66, 16)
  ),
  2 => (
    0 => to_signed(62, 16),
    1 => to_signed(66, 16),
    2 => to_signed(-32, 16),
    3 => to_signed(2, 16),
    4 => to_signed(-7, 16),
    5 => to_signed(4, 16),
    6 => to_signed(-61, 16),
    7 => to_signed(-53, 16),
    8 => to_signed(23, 16)
  ),
  3 => (
    0 => to_signed(-35, 16),
    1 => to_signed(67, 16),
    2 => to_signed(-44, 16),
    3 => to_signed(5, 16),
    4 => to_signed(-20, 16

In [5]:
IMG_W, IMG_H = 12, 98
SCALE_MFCC = 2**15

# À adapter : recharger ton exemple X_val[0] depuis disque si nécessaire
x = X_val[0]          # (98,12,1) ou (98,12)
x2d = x[..., 0] if x.ndim == 3 else x

img_q = np.clip(np.round(x2d * SCALE_MFCC), -32768, 32767).astype(np.int16)

# Sauvegarde pour le TB
np.savetxt("img_q_int16.txt", img_q.reshape(-1), fmt="%d")


In [6]:
# -----------------------------
# Paramètres de la couche
# -----------------------------
IMG_H, IMG_W = 98, 12
H_valid, W_valid = IMG_H - 2, IMG_W - 2
NPIX = H_valid * W_valid

NB_FILTERS = 32              # ton bloc conv3x3_32

# -----------------------------
# 1) Chargement de l'image quantifiée
# -----------------------------
img_q_flat = np.loadtxt("img_q_int16.txt", dtype=np.int16)
img_q = img_q_flat.reshape(IMG_H, IMG_W)   # (98,12)

# On travaille en int32 pour les produits/sommes
img_q_i32 = img_q.astype(np.int32)

# -----------------------------
# 2) Convolution entière pour les 32 premiers filtres
# -----------------------------
out_all = np.zeros((NB_FILTERS, H_valid, W_valid), dtype=np.int32)

for f in range(NB_FILTERS):   # filtres 0..31
    # Charge le noyau f, déjà en Q8.8 (int16)
    ker_q = np.loadtxt(f"conv1_kernel_{f:03d}.txt",
                       dtype=np.int16).reshape(3, 3)
    ker_i32 = ker_q.astype(np.int32)

    # Convolution "valid" 3x3 en entier
    for y in range(1, IMG_H - 1):      # 1..96
        for x in range(1, IMG_W - 1):  # 1..10
            patch = img_q_i32[y-1:y+2, x-1:x+2]   # (3,3)
            out_all[f, y-1, x-1] = np.sum(patch * ker_i32)

# -----------------------------
# 3) Aplatissement comme dans conv3x3_32
#     addr = f * NPIX + (y * W_valid + x)
# -----------------------------
golden_flat = np.zeros(NB_FILTERS * NPIX, dtype=np.int32)

for f in range(NB_FILTERS):
    # reshape en (H_valid*W_valid,) dans l'ordre row-major (y,x)
    fmap_flat = out_all[f].reshape(-1)
    golden_flat[f * NPIX : (f+1) * NPIX] = fmap_flat

# -----------------------------
# 4) Sauvegarde pour le testbench
# -----------------------------
np.savetxt("conv1_bank0_out_q_int32.txt", golden_flat, fmt="%d")

print("Golden conv1, bank0 : shape all =", out_all.shape,
      ", flat =", golden_flat.shape)


Golden conv1, bank0 : shape all = (32, 96, 10) , flat = (30720,)


In [7]:
def export_weight(name, data, width=16, qscale=256):
    """Quantize array `data` by `qscale` then print C-style initializer."""
    q_data = np.round(data * qscale).astype(int)
    q_data = np.clip(q_data, -2**(width-1), 2**(width-1)-1)
    flat = q_data.flatten()
    print(f'const model_t {name}[{len(flat)}] = {'{'}')
    print(', '.join(map(str, flat)))
    print('};\\n')

# Export all layer weights if a Keras model was loaded
if 'model' in globals() and model is not None:
    for i, layer in enumerate(model.layers):
        weights = layer.get_weights()
        if not weights:
            continue
        for j, w in enumerate(weights):
            arr = np.array(w)
            export_weight(f'model_layer{i}_w{j}', arr)
else:
    print('No Keras model loaded; skipping layer export.')

# Informational: shapes of first layer weights (if present)
if 'model' in globals() and model is not None:
    try:
        w0 = model.layers[1].get_weights()
        print('Layer 1 weights shapes:', [a.shape for a in w0])
    except Exception as e:
        print('Could not read layer 1 weights:', e)

# Save a few sample inputs/outputs for target board testing
if 'X_val' in globals() and X_val is not None:
    nsamples = min(10, len(X_val))
    for i in range(nsamples):
        np.savetxt(f'X_sample_{i}.txt', X_val[i].reshape(-1), fmt='%f')
        if 'y_val' in globals() and y_val is not None:
            np.savetxt(f'y_sample_{i}.txt', np.atleast_1d(y_val[i]).reshape(-1), fmt='%d')
    print(f'Saved {nsamples} X/y samples as text files')
elif 'X_val' in globals() and X_val is not None:
    print('X_val available but y_val missing; saved sample image earlier.')


SyntaxError: closing parenthesis ')' does not match opening parenthesis '{' (2796609252.py, line 6)